# QAM receiver quickstart

A 32 GBd 16-QAM link at 2 samples per symbol, end to end: generate, impair,
matched-filter, equalize with inline carrier recovery, measure, and continue
on the next record.  Every stage is a plain function on a `Signal`; the
algorithm choices (`RRC`, `BPS`, `CycleSlip`) are small objects written in the
call.  The `Signal` carries its facts (rates), its description
(constellation, pulse) and the transmitted reference, so later stages read
them instead of taking them again.

In [ ]:
import matplotlib.pyplot as plt

import commkit as ck
from commkit import RRC, Constellation
from commkit.recovery import BPS, CycleSlip

# The device follows the data: build on the CPU, move once if a GPU is there.
DEVICE = "gpu" if ck.backend.is_cupy_available() else "cpu"
DEVICE

## 1. Transmitter

`generate` draws bits with the seeded `rng`, maps them on the constellation
and shapes the pulse.  The result keeps the transmitted symbols and bits in
`tx.reference`.

In [ ]:
tx = ck.generate(
    Constellation.qam(16),
    num_symbols=2**16,
    symbol_rate=32e9,
    sps=2,
    pulse=RRC(rolloff=0.1),
    rng=1,
)
print(tx)
ck.plotting.plot_ideal_constellation(tx.constellation)
plt.show()

## 2. Channel

100 kHz laser phase noise and AWGN at Es/N0 = 18 dB, then an explicit move to
the GPU.  Every later step runs where the samples are.

In [ ]:
rx = ck.impairments.apply_phase_noise(tx, linewidth=100e3, rng=2)
rx = ck.impairments.apply_awgn(rx, esn0_db=18, rng=3).to(DEVICE)
ck.plotting.plot_psd(rx)
plt.show()

## 3. Matched filter

With no `pulse=` argument, `matched_filter` uses the pulse the Signal
carries.

In [ ]:
rx = ck.filtering.matched_filter(rx)
ck.plotting.plot_constellation(ck.multirate.decimate_to_symbol_rate(rx))
plt.show()

Before equalization the phase noise smears every point into an arc.

## 4. Equalizer with inline carrier recovery

A 21-tap LMS at 2 sps with blind phase search inside the loop.  The first
2000 symbols train both the taps and the phase (so no rotation ambiguity
remains); decisions take over after them.  `res.signal` is the 1-SPS output
with the reference aligned to it.

In [ ]:
cpr = BPS(test_phases=64, cycle_slip=CycleSlip(history=100))
res = ck.equalization.lms(
    rx,
    num_taps=21,
    step_size=1e-3,
    training_symbols=rx.reference.symbols[..., :2000],
    cpr=cpr,
)
ck.plotting.plot_equalizer_result(res)
plt.show()

In [ ]:
y = res.signal
ck.plotting.plot_constellation(y, overlay_ideal=True)
plt.show()

## 5. Metrics

Metrics compare against the reference the Signal carries and return host
floats (one per channel for MIMO).  `num_skip_symbols` leaves out the
training and convergence region.

In [ ]:
skip = 2000
snr_db = ck.metrics.snr(y, num_skip_symbols=skip)
print(f"EVM {ck.metrics.evm(y, num_skip_symbols=skip):.1f} %")
print(f"SNR {snr_db:.1f} dB")
print(f"BER {ck.metrics.ber(y, num_skip_symbols=skip):.2e}")
# LLRs need the noise variance on the constellation's (unit-power) scale.
noise_var = 10 ** (-snr_db / 10)
print(
    f"GMI {ck.metrics.gmi(y, noise_var=noise_var, num_skip_symbols=skip):.3f} b/symbol"
)

## 6. The next record

A capture is often processed in records.  `state=res.state` continues from
the converged taps and carrier phase, so the next record needs no training
and has no convergence transient.

In [ ]:
rx_next = ck.impairments.apply_awgn(
    ck.impairments.apply_phase_noise(tx, linewidth=100e3, rng=4), esn0_db=18, rng=5
).to(DEVICE)
rx_next = ck.filtering.matched_filter(rx_next)
res2 = ck.equalization.lms(
    rx_next, num_taps=21, step_size=1e-3, cpr=cpr, state=res.state
)
print(f"EVM (whole record, no training) {ck.metrics.evm(res2.signal):.1f} %")